In [1]:
# For the simplicity of indexing, we use lower triangular matrices. 
# Make sure everything is lower triangular

# We verify here the semi-smallness of the squaring map on B for n <= 5. 
# For n in this range, there are finitely many B-orbits on the nilradical of B, 
# thus the strata dimensions are easy to compute. 
# The dimension of the stratum containing A is dim B - dim Z_B(A) + c 
# where c is a parameter dimension on the number of distinct eigenvalues
# The dimension of the fiber is dim Z_B(A) -  dim Z_B(M)

# To check semi-smallness, we compute the quantity dim B + dim Z_B(A) - 2 dim Z_B(M) + c <= dim B?
# Thus semi-smallness is satisfied if and only if dim Z_B(A) <= 2 dim Z_B(M) - c
# the defect number is 2 dim Z_B(M) - dim Z_B(A) - c
# We verify this inequality for all n \leq 5 and orbit representatives M (with redundancies)
# Iterating on M is the easiest way to do this.

# There is an interesting tangent space argument showing finiteness of the Z_B(A) orbits in the
# fiber of the squaring map.

In [2]:
def centralizer_dimension(A):
    # A is a lower triangular matrix
    # This function computes the dimension of the stabilizer of A under B-action
    
    n = A.nrows()
    if n != A.ncols():
        raise ValueError("The matrix A must be a square matrix")

    # might be too slow to check lower triangular

    F = A.base_ring()

    # dimension of Lie(B)
    dim_domain = n*(n + 1) // 2

    basis_image = []
    for i in range(n):
        for j in range(i + 1):
            E = matrix(F, n, n)
            E[i, j] = 1

            commutator = E * A - A * E

            basis_image.append(commutator.list())

    d_A = matrix(F, basis_image)

    dim_ker = dim_domain - d_A.rank()

    return dim_ker

In [3]:
# An example computing the centralizer

F = QQ;
A = matrix([[0, 0, 0], [0, 0, 0], [1, 0, 0]])

centralizer_dimension(A)

5

In [4]:
# When iterating M, we iterate over a set-partition of the diagonal of M
# Since M = Ms * Mu, we may first conjugate Ms to a diagonal matrix
# Then Mu will be conjugated to a generalized "block matrix" (a permuted version of a block matrix)
# Since each block will have size <= 5, Mu will have 0-1 entries 
# up to the conjugation by the centralizer of Ms.
# Thus to iterate M we just need to iterate a 0-1 entried Mn, and a diagonal Ms which commutes with Mn

# To do this, for each 0-1 entried Mn, we iterate over all representative candidates of Ms

In [5]:
import itertools

# The following is how we iterate over candidates of Ms
def unordered_partition_diagonal_matrices(S):
    """
    Yields diagonal matrices whose diagonals are linear combinations 
    of vectors in S based on unordered set partitions.
    """
    elements = list(S)
    
    for p in SetPartitions(elements):
        term_choices = []
        
        for i, part in enumerate(list(p)):
            coeff_mag = i + 1  
            
            # Cast the part to a list to fix an arbitrary but consistent order
            part_list = list(part) 
            
            # 1. The first vector in the subset gets ONLY a positive coefficient
            term_choices.append([coeff_mag * part_list[0]])
            
            # 2. All remaining vectors in the subset get BOTH positive and negative
            for v in part_list[1:]:
                term_choices.append([coeff_mag * v, -coeff_mag * v])
                
        # Generate the newly reduced sign combinations
        for combo in itertools.product(*term_choices):
            # Sum, convert to diagonal matrix, and yield
            yield diagonal_matrix(sum(combo)), len(p) # len(p) is the number of continuous parameters

In [6]:
# An exampel demonstrating the iteration scheme
# The output diagonal matrix is one possible candidate of Ms for the trivial Mn
# The number next to the diagonal matrix is the number of free parameters for eigenvalues of this Ms

# Define a vector space and a set S of vectors
V = VectorSpace(QQ, 3)
v1 = V([1, 0, 0])
v2 = V([0, 1, 0])
v3 = V([0, 0, 1])

S = [v1, v2, v3]

for v in S:
    v.set_immutable()

# Create the generator
matrices = unordered_partition_diagonal_matrices(S)

# Iterate and print the results
print(f"Vectors in S: {S}\n")
for mat in matrices:
    print(mat)
    print("---") # Visual separator for the matrices

Vectors in S: [(1, 0, 0), (0, 1, 0), (0, 0, 1)]

([1 0 0]
[0 1 0]
[0 0 1], 1)
---
([ 1  0  0]
[ 0 -1  0]
[ 0  0  1], 1)
---
([ 1  0  0]
[ 0  1  0]
[ 0  0 -1], 1)
---
([ 1  0  0]
[ 0 -1  0]
[ 0  0 -1], 1)
---
([2 0 0]
[0 1 0]
[0 0 1], 2)
---
([ 2  0  0]
[ 0 -1  0]
[ 0  0  1], 2)
---
([1 0 0]
[0 2 0]
[0 0 1], 2)
---
([ 1  0  0]
[ 0  2  0]
[ 0  0 -1], 2)
---
([2 0 0]
[0 2 0]
[0 0 1], 2)
---
([ 2  0  0]
[ 0 -2  0]
[ 0  0  1], 2)
---
([3 0 0]
[0 2 0]
[0 0 1], 3)
---


In [7]:
def check(Mn):
    # check the semi-smallness of fibers involving Mn
    n = Mn.nrows()
    if n != Mn.ncols():
        raise ValueError

    # might be too slow to check lower triangular or 0-1 entried
    # compute a basis for the diagonal Ms
    F = Mn.base_ring()
    basis_images = []
    for i in range(n):
        E = matrix(F, n, n)
        E[i, i] = 1

        commutator = E * Mn - Mn * E
        basis_images.append(commutator.list())

    d_Mn = matrix(F, basis_images) # images are row vectors
    Ms_basis = d_Mn.kernel().basis() # basis here correspond to some set partitions

    for Ms, c in unordered_partition_diagonal_matrices(Ms_basis):
        M = Ms + Mn
        A = M*M

        ZBA = centralizer_dimension(A)
        ZBM = centralizer_dimension(M)

        if ZBA == 2* ZBM - c:
            # When the stratum reaches the upper limit
            print(M, "\n")

        if ZBA > 2 * ZBM - c:
            # when semi-smallness is broken
            print("FALSE")
            return False

    # if passed all Ms check for this Mn
    return True

In [8]:
# An example checking all 3x3 M with trivial Mn
N = matrix([[0, 0, 0], [0, 0, 0], [0, 0, 0]])
check(N)

[ 2  0  0]
[ 0 -1  0]
[ 0  0  1] 

[ 1  0  0]
[ 0  2  0]
[ 0  0 -1] 

[ 2  0  0]
[ 0 -2  0]
[ 0  0  1] 

[3 0 0]
[0 2 0]
[0 0 1] 



True

In [9]:
# Another 3x3 example
N = matrix([[0, 0, 0], [0, 0, 0], [1, 0, 0]])
check(N)

[ 1  0  0]
[ 0 -1  0]
[ 1  0  1] 



True

In [10]:
# The above example outputs an interesting stratum. Let's verify this by hand.

M = matrix([[1, 0, 0], [0, -1, 0], [1, 0, 1]])

print(centralizer_dimension(M)) # 3
print(centralizer_dimension(M*M)) # 5

# The strata containing A = M^2 should have dimension 2 (conjugation orbit + eigenvalue freedom)
# While the fiber has dimension 5 - 3 = 2, thus this stratum reaches the upper limit.

3
5


In [11]:
# An iterator generating all representative lower triangular nilpotent matrices
# It is known any such representative has entries 0, 1

def lower_triangular_generator(n, F):
    if n <= 1:
        yield matrix(F, n, n) # the zero matrix
        return

    num_entries = n * (n - 1)//2
    binaries = itertools.product([0, 1], repeat=num_entries) # lazy binary generation

    for binary in binaries:
        M = matrix(F, n, n)
        idx = 0
        for i in range(n):
            for j in range(i):
                M[i, j] = binary[idx]
                idx += 1
        yield M

In [12]:
# the final check
for n in range(1, 6):
    for Mn in lower_triangular_generator(n, QQ):
        check(Mn)

[1] 

[-1  0]
[ 0  1] 

[2 0]
[0 1] 

[ 2  0  0]
[ 0 -1  0]
[ 0  0  1] 

[ 1  0  0]
[ 0  2  0]
[ 0  0 -1] 

[ 2  0  0]
[ 0 -2  0]
[ 0  0  1] 

[3 0 0]
[0 2 0]
[0 0 1] 

[ 1  0  0]
[ 0 -1  0]
[ 1  0  1] 

[-2  0  0  0]
[ 0  2  0  0]
[ 0  0 -1  0]
[ 0  0  0  1] 

[ 3  0  0  0]
[ 0  2  0  0]
[ 0  0 -1  0]
[ 0  0  0  1] 

[-2  0  0  0]
[ 0 -1  0  0]
[ 0  0  2  0]
[ 0  0  0  1] 

[ 3  0  0  0]
[ 0 -1  0  0]
[ 0  0  2  0]
[ 0  0  0  1] 

[-1  0  0  0]
[ 0  2  0  0]
[ 0  0 -2  0]
[ 0  0  0  1] 

[ 3  0  0  0]
[ 0  2  0  0]
[ 0  0 -2  0]
[ 0  0  0  1] 

[-1  0  0  0]
[ 0  3  0  0]
[ 0  0  2  0]
[ 0  0  0  1] 

[-2  0  0  0]
[ 0  3  0  0]
[ 0  0  2  0]
[ 0  0  0  1] 

[-3  0  0  0]
[ 0  3  0  0]
[ 0  0  2  0]
[ 0  0  0  1] 

[4 0 0 0]
[0 3 0 0]
[0 0 2 0]
[0 0 0 1] 

[ 2  0  0  0]
[ 0  1  0  0]
[ 0  0 -1  0]
[ 0  1  0  1] 

[-1  0  0  0]
[ 0  1  0  0]
[ 0  0  1  0]
[ 1  0  0 -1] 

[-1  0  0  0]
[ 0  2  0  0]
[ 0  0  1  0]
[ 1  0  0 -1] 

[-2  0  0  0]
[ 0  2  0  0]
[ 0  0  1  0]
[ 1  0  0 -2] 



In [13]:
# All strata passed the semi-smallness test. 
# However the strata which hit the upper bound do not appear to follow certain pattern
# Could semi-smallness be broken for n \geq 6? 
# In these cases the stratum dimension should be larger than expected from normal forms

In [14]:
# Some more experiments:
# Can we get a glimpse of the centralizer dimension stratification? 
# The following attemps to list the normal forms which share the same centralizer dimension

In [15]:
def unsigned_unordered_partition_diagonal_matrices(S):
    elements = list(S)
    
    for p in SetPartitions(elements):
        # term_choices = []
        combo = []
        
        for i, part in enumerate(list(p)):
            coeff_mag = i + 1  
            
            # Cast the part to a list to fix an arbitrary but consistent order
            part_list = list(part) 
            
            # 1. The first vector in the subset gets ONLY a positive coefficient
            # term_choices.append([coeff_mag * part_list[0]])
            
            # 2. All remaining vectors in the subset get BOTH positive and negative
            for v in part_list:
                combo.append(coeff_mag * v)
                
        # Generate the newly reduced sign combinations
        # for combo in itertools.product(*term_choices):
            # Sum, convert to diagonal matrix, and yield
        yield diagonal_matrix(sum(combo))

In [16]:
# Define a vector space and a set S of vectors
V = VectorSpace(QQ, 3)
v1 = V([1, 0, 0])
v2 = V([0, 1, 0])
v3 = V([0, 0, 1])

S = [v1, v2, v3]

for v in S:
    v.set_immutable()

# Create the generator
matrices = unsigned_unordered_partition_diagonal_matrices(S)

# Iterate and print the results
print(f"Vectors in S: {S}\n")
for mat in matrices:
    print(mat)
    print("---") # Visual separator for the matrices

Vectors in S: [(1, 0, 0), (0, 1, 0), (0, 0, 1)]

[1 0 0]
[0 1 0]
[0 0 1]
---
[2 0 0]
[0 1 0]
[0 0 1]
---
[1 0 0]
[0 2 0]
[0 0 1]
---
[2 0 0]
[0 2 0]
[0 0 1]
---
[3 0 0]
[0 2 0]
[0 0 1]
---


In [17]:
def centralizer_stratification(n):
    if n > 5:
        raise ValueError("n is too big, should be less or equal to 5")

    centralizer_dim = dict()
    for i in range(1, n*(n + 1)//2 + 1):
        centralizer_dim[i] = []

    for Mn in lower_triangular_generator(n, QQ):
        n = Mn.nrows()
        if n != Mn.ncols():
            raise ValueError
    
        # might be too slow to check lower triangular or 0-1 entried
        # compute a basis for the diagonal Ms
        F = Mn.base_ring()
        basis_images = []
        for i in range(n):
            E = matrix(F, n, n)
            E[i, i] = 1
    
            commutator = E * Mn - Mn * E
            basis_images.append(commutator.list())
    
        d_Mn = matrix(F, basis_images) # images are row vectors
        Ms_basis = d_Mn.kernel().basis() # basis here correspond to some set partitions
    
        for Ms in unsigned_unordered_partition_diagonal_matrices(Ms_basis):
            M = Ms + Mn
            centralizer_dim[centralizer_dimension(M)].append(M) 
    
    return centralizer_dim

In [18]:
centralizer_stratification(4)

{1: [],
 2: [],
 3: [],
 4: [
[4 0 0 0]  [3 0 0 0]  [3 0 0 0]  [3 0 0 0]  [3 0 0 0]  [2 0 0 0]
[0 3 0 0]  [0 2 0 0]  [0 2 0 0]  [0 2 0 0]  [0 2 0 0]  [0 1 0 0]
[0 0 2 0]  [0 0 1 0]  [0 0 1 0]  [0 0 1 0]  [0 1 2 0]  [0 1 1 0]
[0 0 0 1], [0 0 1 1], [0 1 0 2], [1 0 0 3], [0 0 0 1], [0 0 1 1],

[2 0 0 0]  [2 0 0 0]  [3 0 0 0]  [2 0 0 0]  [2 0 0 0]  [2 0 0 0]
[0 1 0 0]  [0 1 0 0]  [0 2 0 0]  [0 1 0 0]  [0 1 0 0]  [0 1 0 0]
[0 1 1 0]  [0 1 1 0]  [1 0 3 0]  [1 0 2 0]  [1 0 2 0]  [1 0 2 0]
[0 1 1 1], [1 0 0 2], [0 0 0 1], [0 0 1 2], [0 1 0 1], [1 0 1 2],

[3 0 0 0]  [2 0 0 0]  [2 0 0 0]  [2 0 0 0]  [2 0 0 0]  [1 0 0 0]
[1 3 0 0]  [1 2 0 0]  [1 2 0 0]  [1 2 0 0]  [1 2 0 0]  [1 1 0 0]
[0 0 2 0]  [0 0 1 0]  [0 0 1 0]  [0 0 1 0]  [0 1 2 0]  [0 1 1 0]
[0 0 0 1], [0 0 1 1], [0 1 0 2], [1 1 0 2], [0 0 0 1], [0 0 1 1],

[1 0 0 0]  [1 0 0 0]  [1 0 0 0]  [2 0 0 0]  [1 0 0 0]  [1 0 0 0]
[1 1 0 0]  [1 1 0 0]  [1 1 0 0]  [1 2 0 0]  [1 1 0 0]  [1 1 0 0]
[0 1 1 0]  [0 1 1 0]  [0 1 1 0]  [1 1 2 0]  [1 1 1 0] 

In [19]:
centralizer_stratification(5)

{1: [],
 2: [],
 3: [],
 4: [],
 5: [
[5 0 0 0 0]  [4 0 0 0 0]  [4 0 0 0 0]  [4 0 0 0 0]  [4 0 0 0 0]
[0 4 0 0 0]  [0 3 0 0 0]  [0 3 0 0 0]  [0 3 0 0 0]  [0 3 0 0 0]
[0 0 3 0 0]  [0 0 2 0 0]  [0 0 2 0 0]  [0 0 2 0 0]  [0 0 2 0 0]
[0 0 0 2 0]  [0 0 0 1 0]  [0 0 0 1 0]  [0 0 0 1 0]  [0 0 0 1 0]
[0 0 0 0 1], [0 0 0 1 1], [0 0 1 0 2], [0 1 0 0 3], [1 0 0 0 4],

[4 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]
[0 3 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]
[0 0 2 0 0]  [0 0 1 0 0]  [0 0 1 0 0]  [0 0 1 0 0]  [0 0 1 0 0]
[0 0 1 2 0]  [0 0 1 1 0]  [0 0 1 1 0]  [0 0 1 1 0]  [0 0 1 1 0]
[0 0 0 0 1], [0 0 0 1 1], [0 0 1 1 1], [0 1 0 0 2], [1 0 0 0 3],

[4 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]  [3 0 0 0 0]
[0 3 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]  [0 2 0 0 0]
[0 0 2 0 0]  [0 0 1 0 0]  [0 0 1 0 0]  [0 0 1 0 0]  [0 0 1 0 0]
[0 1 0 3 0]  [0 1 0 2 0]  [0 1 0 2 0]  [0 1 0 2 0]  [0 1 0 2 0]
[0 0 0 0 1], [0 0 0 1 2], [0 0 1 0 1], [0 1 0 1 2], [1 0 0 0 3